In [1]:
from kaggle_secrets import UserSecretsClient
import os          
us = UserSecretsClient()  
os.environ["KAGGLE_KEY"] = us.get_secret("KAGGLE_KEY")
os.environ["KAGGLE_USERNAME"] = us.get_secret("KAGGLE_USERNAME") 

In [2]:
!pip install -q diffusers transformers accelerate safetensors  
!pip install -q xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 11.7 MB/s eta 0:00:00


In [3]:
"""
FULL prompt-based debiasing -- PixArt-Sigma, memory-safe, no OOM.
RANGE-LIMITED FIX: only regenerates prompt_id 0..251 (inclusive), the range
that lost its images earlier. Still rebuilds manifest from ACTUAL files on
disk before running, so it can't be fooled by a stale manifest.csv again.

RUN ON FRESH SESSION (Restart & Clear), no other model loaded.
  !pip install -q diffusers transformers accelerate safetensors
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, csv, json, glob, subprocess, shutil
import pandas as pd, torch
from diffusers import PixArtSigmaPipeline

# ---------------------------------------------------------------------------
# Double-check these against your actual Kaggle "Input" panel before running.
# ---------------------------------------------------------------------------
XLSX_MOUNT   = "/kaggle/input/datasets/ajfaisal002/t2i-xlxs"
INPUT_XLSX   = os.path.join(XLSX_MOUNT, "ProfessionAdjective Bias.xlsx")
OUTPUT_ROOT  = "/kaggle/working/pixart_debiased_prompt"
DATASET_SLUG = f"{os.environ.get('KAGGLE_USERNAME','ajfaisal002')}/pixart-debiased-prompt"
RESTORE_MOUNT = f"/kaggle/input/datasets/{os.environ.get('KAGGLE_USERNAME','ajfaisal002')}/pixart-debiased-prompt"

# ---- the only range we want to touch this run ----
PID_START, PID_END = 0, 251   # inclusive

IMAGES_PER_PROMPT = 50       # 25 male + 25 female
STEPS, GUIDANCE   = 6, 4.0
HEIGHT = WIDTH    = 512
PUSH_EVERY_N      = 5
STYLE_SUFFIX = ("realistic photograph, photorealistic, professional photography, "
                "natural lighting, high detail, DSLR photo")
DEVICE = "cuda"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# resume restore (copies any files from a previously pushed dataset version)
if os.path.isdir(RESTORE_MOUNT):
    restored = 0
    for dp, _d, fs in os.walk(RESTORE_MOUNT):
        rel = os.path.relpath(dp, RESTORE_MOUNT)
        dd = os.path.join(OUTPUT_ROOT, rel) if rel != "." else OUTPUT_ROOT
        os.makedirs(dd, exist_ok=True)
        for f in fs:
            s, d = os.path.join(dp, f), os.path.join(dd, f)
            if not os.path.exists(d):
                shutil.copy2(s, d); restored += 1
    print(f"Restored {restored} prior files." if restored else "No prior progress found at RESTORE_MOUNT.")
else:
    print(f"[warn] RESTORE_MOUNT does not exist: {RESTORE_MOUNT} -- nothing restored this session.")

df = pd.read_excel(INPUT_XLSX).reset_index().rename(columns={"index": "prompt_id"})
neutral_all = df[df["category"].astype(str).str.lower() == "neutral"].reset_index(drop=True)
print(f"{len(neutral_all)} neutral prompts total.")

# ---- filter down to just the broken range ----
neutral = neutral_all[(neutral_all["prompt_id"] >= PID_START) & (neutral_all["prompt_id"] <= PID_END)].reset_index(drop=True)
print(f"Restricting this run to prompt_id {PID_START}..{PID_END} -> {len(neutral)} prompts.")

pipe = PixArtSigmaPipeline.from_pretrained(
    "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS", torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()     # the ONLY way PixArt fits a T4
pipe.enable_attention_slicing()
pipe.set_progress_bar_config(disable=True)

def make_variant(prompt, gw):
    p = prompt.strip()
    for art in ("A ", "An ", "a ", "an "):
        if p.startswith(art): return f"{art}{gw} {p[len(art):]}"
    return f"{gw} {p}"

MANIFEST = os.path.join(OUTPUT_ROOT, "manifest.csv")
METADATA = os.path.join(OUTPUT_ROOT, "metadata.csv")

# ---------------------------------------------------------------------------
# Rebuild done_map from files that ACTUALLY exist on disk for the FULL
# neutral set (not just the restricted range), so the manifest.csv we write
# stays correct/consistent for the whole dataset, not just this run's slice.
# ---------------------------------------------------------------------------
def rebuild_manifest_from_disk(neutral_df):
    rows = []
    done_map = {}
    for _, row in neutral_df.iterrows():
        pid = row.prompt_id
        prof = str(row.profession).replace(" ", "_")
        for variant in ("male", "female"):
            folder = os.path.join(OUTPUT_ROOT, "pixart_sigma", prof, variant)
            n_files = len(glob.glob(os.path.join(folder, "img_*.png"))) if os.path.isdir(folder) else 0
            rows.append([pid, row.profession, variant, row.prompt, n_files])
            done_map[(pid, variant)] = n_files
    man_df = pd.DataFrame(rows, columns=["prompt_id", "profession", "variant", "prompt", "done"])
    man_df.to_csv(MANIFEST, index=False)
    return man_df, done_map

man, done_map = rebuild_manifest_from_disk(neutral_all)
n_missing_in_range = sum(1 for pid in neutral["prompt_id"] for v in ("male","female")
                          if done_map.get((pid, v), 0) < IMAGES_PER_PROMPT // 2)
print(f"[manifest rebuilt from disk] {n_missing_in_range} (prompt_id, variant) pairs still need images in range {PID_START}-{PID_END}.")

META_FIELDS = ["model","method","prompt_id","profession","pushed_gender",
               "base_prompt","full_prompt","image_file"]
if not os.path.exists(METADATA):
    with open(METADATA, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=META_FIELDS).writeheader()

def save_man(): man.to_csv(MANIFEST, index=False)
def append_meta(rows):
    with open(METADATA, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=META_FIELDS).writerows(rows)

def push(msg="ckpt"):
    meta = os.path.join(OUTPUT_ROOT, "dataset-metadata.json")
    if not os.path.exists(meta):
        json.dump({"title": DATASET_SLUG.split("/")[-1],"id":DATASET_SLUG,
                   "licenses":[{"name":"CC0-1.0"}]}, open(meta,"w"))
    r = subprocess.run(["kaggle","datasets","version","-p",OUTPUT_ROOT,"-m",msg],
                       capture_output=True, text=True)
    if r.returncode != 0:
        r = subprocess.run(["kaggle","datasets","create","-p",OUTPUT_ROOT],
                           capture_output=True, text=True)
    print(f"[ckpt] pushed: {msg}" if r.returncode==0 else f"[ckpt] FAIL\n{r.stderr}")

HALF = IMAGES_PER_PROMPT // 2
psp = 0
with torch.inference_mode():
    for _, row in neutral.iterrows():          # <-- only the restricted 0..251 range
        pid = row.prompt_id; prof = str(row.profession).replace(" ", "_")
        base_prompt = row.prompt
        for variant, gw in [("male","male"), ("female","female")]:
            already = done_map.get((pid, variant), 0)
            if already >= HALF: continue
            full_prompt = f"{make_variant(base_prompt, gw)} {STYLE_SUFFIX}"

            folder = os.path.join(OUTPUT_ROOT, "pixart_sigma", prof, variant)
            os.makedirs(folder, exist_ok=True)
            ic = already
            print(f"[{pid}] {make_variant(base_prompt, gw)} ({already}/{HALF})")
            for i in range(already, HALF):
                gen = torch.Generator(DEVICE).manual_seed(1000*pid + (0 if variant=="male" else 500) + i)
                try:
                    img = pipe(full_prompt,
                               num_inference_steps=STEPS, guidance_scale=GUIDANCE,
                               height=HEIGHT, width=WIDTH, clean_caption=False,
                               generator=gen).images[0]
                    ic += 1; fp = f"{folder}/img_{ic}.png"; img.save(fp, dpi=(300,300))
                    append_meta([{"model":"pixart_sigma","method":"prompt","prompt_id":pid,
                                  "profession":row.profession,"pushed_gender":variant,
                                  "base_prompt":base_prompt,"full_prompt":full_prompt,"image_file":fp}])
                    del img; gc.collect(); torch.cuda.empty_cache()
                except RuntimeError as e:
                    print(f"[warn] {pid} {variant} img {ic+1}: {e}")
                    torch.cuda.empty_cache(); continue
                man.loc[(man.prompt_id==pid)&(man.variant==variant),"done"]=ic
                save_man()
            psp += 1
            if psp >= PUSH_EVERY_N: push(f"pixart prompt {pid} {variant} (backfill 0-251)"); psp = 0

push("final pixart prompt-debias backfill 0-251")
print(f"\nDone: backfilled prompt_id {PID_START}-{PID_END}.")
print(f"Metadata CSV: {METADATA}")
print("\nZipping...")
zp = shutil.make_archive("/kaggle/working/pixart_debiased_prompt_FINAL", "zip", OUTPUT_ROOT)
print(f"Created: {zp} ({os.path.getsize(zp)/(1024*1024):.1f} MB)")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Restored 2 prior files.
20 neutral prompts total.
Restricting this run to prompt_id 0..251 -> 12 prompts.


model_index.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

[manifest rebuilt from disk] 24 (prompt_id, variant) pairs still need images in range 0-251.
[0] A male doctor at work. (0/25)
[0] A female doctor at work. (0/25)
[21] A male nurse at work. (0/25)
[21] A female nurse at work. (0/25)
[42] A male engineer at work. (0/25)
[ckpt] pushed: pixart prompt 42 male (backfill 0-251)
[42] A female engineer at work. (0/25)
[63] A male teacher at work. (0/25)
[63] A female teacher at work. (0/25)
[84] A male scientist at work. (0/25)
[84] A female scientist at work. (0/25)
[ckpt] pushed: pixart prompt 84 female (backfill 0-251)
[105] A male lawyer at work. (0/25)
[105] A female lawyer at work. (0/25)
[126] A male ceo at work. (0/25)
[126] A female ceo at work. (0/25)
[147] A male police officer at work. (0/25)
[ckpt] pushed: pixart prompt 147 male (backfill 0-251)
[147] A female police officer at work. (0/25)
[168] A male pilot at work. (0/25)
[168] A female pilot at work. (0/25)
[189] A male mechanic at work. (0/25)
[189] A female mechanic at work.

In [4]:
import os, shutil

# folder that holds pixart_sigma, metadata.csv, manifest.csv, etc.
OUTPUT_ROOT = "/kaggle/working/pixart_debiased_prompt"

# create the zip in /kaggle/working so it shows up in the Output panel
zip_path = shutil.make_archive(
    "/kaggle/working/pixart_debiased_prompt_FINAL2",  # output name (no .zip)
    "zip",
    OUTPUT_ROOT
)

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"Created: {zip_path} ({size_mb:.1f} MB)")

# quick sanity check of what's inside
import zipfile
with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
print(f"Total entries in zip: {len(names)}")
print("Sample:", names[:10])

Created: /kaggle/working/pixart_debiased_prompt_FINAL2.zip (190.1 MB)
Total entries in zip: 640
Sample: ['pixart_sigma/', 'manifest.csv', 'dataset-metadata.json', 'metadata.csv', 'pixart_sigma/CEO/', 'pixart_sigma/Chef/', 'pixart_sigma/Doctor/', 'pixart_sigma/Engineer/', 'pixart_sigma/Journalist/', 'pixart_sigma/Lawyer/']


In [5]:
import os

path="/kaggle/working/pixart_debiased_prompt_FINAL2.zip"

print(os.path.exists(path))
print(os.path.isfile(path))
print(os.path.getsize(path)/(1024*1024), "MB")

True
True
190.12807941436768 MB


In [6]:
!ls -lh /kaggle/working

total 381M
---------- 1 root root  28K Aug  4 23:19 __notebook__.ipynb
drwxr-xr-x 3 root root 4.0K Aug  4 20:09 pixart_debiased_prompt
-rw-r--r-- 1 root root 191M Aug  4 23:19 pixart_debiased_prompt_FINAL2.zip
-rw-r--r-- 1 root root 191M Aug  4 23:19 pixart_debiased_prompt_FINAL.zip
